# Nemotron-OCR-v1
Nemotron-OCR-v1 is a state-of-the-art OCR model developed by NVIDIA, designed for high-accuracy text extraction from complex documents and natural scene images.

In [1]:
!git lfs install
!git clone https://huggingface.co/nvidia/nemotron-ocr-v1

Git LFS initialized.
Cloning into 'nemotron-ocr-v1'...
remote: Enumerating objects: 370, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 370 (delta 0), reused 0 (delta 0), pack-reused 367 (from 1)
Receiving objects: 100% (370/370), 277.25 KiB | 4.62 MiB/s, done.
Resolving deltas: 100% (166/166), done.


In [ ]:
%cd nemotron-ocr-v1/nemotron-ocr
!pip install hatchling
!pip install -v .

In [1]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 110.3 MB/s eta 0:00:00


### Load model

In [4]:
from nemotron_ocr.inference.pipeline import NemotronOCR
import os

# Set path to checkpoints
checkpoint_path = "/content/nemotron-ocr-v1/checkpoints"

print("Loading model...")
ocr = NemotronOCR(model_dir=checkpoint_path)
print(" Model loaded")

Loading model...
 Model loaded


### Load PDF

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload() 
    pdf_path = list(uploaded.keys())[0]
    print("Using PDF:", pdf_path)

except ImportError:
    pdf_path = "data/benchmark/flexqube_arsredovisning_2022.pdf" 
    
    print(f"Running locally, using: {pdf_path}")

Upload your PDF:


Saving FlexQube.pdf to FlexQube.pdf
PDF uploaded: FlexQube.pdf


### PDF to images

In [6]:
import fitz
from PIL import Image

def pdf_to_images(pdf_path: str, dpi: int = 200) -> list[Image.Image]:
    """Convert PDF pages to PIL Images."""
    doc = fitz.open(pdf_path)
    images = []

    zoom = dpi / 72
    matrix = fitz.Matrix(zoom, zoom)

    for page_num in range(len(doc)):
        page = doc[page_num]
        pix = page.get_pixmap(matrix=matrix)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        images.append(img)
        print(f"  Page {page_num + 1}: {pix.width}x{pix.height}px")

    doc.close()
    return images

print(f"Converting {pdf_path} to images...")
pages = pdf_to_images(pdf_path, dpi=200)
print(f"Converted {len(pages)} pages")

Converting FlexQube.pdf to images...
  Page 1: 1654x2339px
  Page 2: 3308x2339px
  Page 3: 3308x2339px
  Page 4: 3308x2339px
  Page 5: 3308x2339px
  Page 6: 3308x2339px
  Page 7: 3308x2339px
  Page 8: 3308x2339px
  Page 9: 3308x2339px
  Page 10: 3308x2339px
  Page 11: 3308x2339px
  Page 12: 3308x2339px
  Page 13: 3308x2339px
  Page 14: 3308x2339px
  Page 15: 3308x2339px
  Page 16: 3308x2339px
  Page 17: 3308x2339px
  Page 18: 3308x2339px
  Page 19: 3308x2339px
  Page 20: 3308x2339px
  Page 21: 3308x2339px
  Page 22: 3308x2339px
  Page 23: 3308x2339px
  Page 24: 3308x2339px
  Page 25: 3308x2339px
  Page 26: 3308x2339px
  Page 27: 3308x2339px
  Page 28: 3308x2339px
  Page 29: 3308x2339px
  Page 30: 3308x2339px
  Page 31: 3308x2339px
  Page 32: 3308x2339px
  Page 33: 3308x2339px
  Page 34: 3308x2339px
  Page 35: 3308x2339px
  Page 36: 3308x2339px
  Page 37: 3308x2339px
  Page 38: 3308x2339px
  Page 39: 3308x2339px
  Page 40: 3308x2339px
  Page 41: 3308x2339px
  Page 42: 3308x2339px
  Page

### Run OCR on all pages

In [7]:
def ocr_single_page(image: Image.Image, page_num: int) -> str:
    """Run OCR on a single page."""
    temp_path = f"/tmp/page_{page_num}.png"
    image.save(temp_path)

    predictions = ocr(temp_path, merge_level="paragraph")
    text = "\n".join([pred['text'] for pred in predictions])
    return text

print("Running Nemotron-OCR...")
results = []

for i, page in enumerate(pages):
    print(f"Processing page {i + 1}/{len(pages)}...", flush=True)
    text = ocr_single_page(page, i)
    results.append(text)

ocr_result = "\n\n---\n\n".join(results)
print(f"Done! ({len(ocr_result):,} chars)")

Running Nemotron-OCR...
Processing page 1/68...
Processing page 2/68...
Processing page 3/68...
Processing page 4/68...
Processing page 5/68...
Processing page 6/68...
Processing page 7/68...
Processing page 8/68...
Processing page 9/68...
Processing page 10/68...
Processing page 11/68...
Processing page 12/68...
Processing page 13/68...
Processing page 14/68...
Processing page 15/68...
Processing page 16/68...
Processing page 17/68...
Processing page 18/68...
Processing page 19/68...
Processing page 20/68...
Processing page 21/68...
Processing page 22/68...
Processing page 23/68...
Processing page 24/68...
Processing page 25/68...
Processing page 26/68...
Processing page 27/68...
Processing page 28/68...
Processing page 29/68...
Processing page 30/68...
Processing page 31/68...
Processing page 32/68...
Processing page 33/68...
Processing page 34/68...
Processing page 35/68...
Processing page 36/68...
Processing page 37/68...
Processing page 38/68...
Processing page 39/68...
Processing

In [10]:
# Preview
print("OCR Output Preview (first 3000 chars):")
print("="*50)
print(ocr_result[:6000])


OCR Output Preview (first 3000 chars):
Årsredovisning
2022
Cockpit
7
Go 3o Statoo Procuse ?
FLEXQUBE

---

Innehåll
Det här är FlexQube
10
12
20
FlexQube är en global leverantör av modulära
och robusta mekaniska vagnar och robotiserade lösningar för materialhantering. Koncernen
grundades 2010 och har sedan dess säkrat ett stort antal framstående företag som kunder.
TiaaCube âr ctt toknikholeg med husakkkntto
Tysklend bot Englandi Rollege arverksant inom vagnshasarad mener allhnntringg gerom en peterterat mockulkoncept FexCCube utvecker och designer kun danpessde losningar for bácle sebotserad och mekenneered lagnelogssu
Genom freeagels egenulieedllue euh un ka
Milomaaarrkkknneee erbuus rouuata och
galvebrande rebezvegna" FexQube har over 1803
Runcler S7 lencer dar cle prim ara marenadema ar Nordemeakk ach Europa.
TexClbbes kunder arentinns inoi blene
onnat ti werkningdinduatric dicrinut on: nch
hegreerksanhettr Exemppl ph stême kuncer fr eela, Amezon. Veive Cars, Siemens, Auta Scania, 

In [9]:
# Save
output_path = "nemotron_ocr_output.md"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(ocr_result)

print(f"\n Saved to: {output_path}")
files.download(output_path)


 Saved to: nemotron_ocr_output.md


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>